In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import logging
from pathlib import Path

import mne 

In [ ]:
# =============================================================================
# Configuration & constants
# =============================================================================

# Logging format: timestamp - level - message
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# === Paths ===
CURDIR = Path.cwd()
PATH_PREPROCESS_DATA_BIDS = CURDIR.parent.parent / '3_Data' / 'preprocessData' / 'CPP_low-level-1_Manning_et_al_2021'
# directory to save temp data
PATH_RESULTS_EPOCH = CURDIR / 'temp_data' / 'epoch'
PATH_RESULTS_ERP = CURDIR / 'temp_data' / 'erp'
PATH_RESULTS_JOINT = CURDIR / 'temp_data' / 'joint-modeling'

PATH_RESULTS_EPOCH.mkdir(parents=True, exist_ok=True)
PATH_RESULTS_ERP.mkdir(parents=True, exist_ok=True)
PATH_RESULTS_JOINT.mkdir(parents=True, exist_ok=True)


SAMPLING_RATE = 500
CHANNEL_INDICES = [54, 36, 86]  # channels of interest for CPP

ERP_TMIN, ERP_TMAX = -0.6, 0.2
#RESP_PRE, RESP_POST = -0.6, 0.2
BASELINE_TMIN, BASELINE_TMAX = -0.6, -0.4
T_AMS_SLPS_START, T_AMS_SLPS_END = -0.18, -0.08
T_PAMS_START, T_PAMS_END = -0.05, 0.05

SUBJECT_IDS = [f'{i:03d}' for i in range(1,21)]

SUBJECT_IDS_EEG = PATH_PREPROCESS_DATA_BIDS.glob('sub-*/eeg/*.vhdr')
SUBJECT_IDS_BEH = PATH_PREPROCESS_DATA_BIDS.glob('sub-*/beh/*_beh.tsv')

In [ ]:
# =============================================================================
#  Epoch ERP and extract features for joint modeling
# =============================================================================

def smooth(df, sample_rate=500, window_ms=200):
    """Simple moving-average smoothing for trial x time data."""
    win = max(1, int(sample_rate * window_ms / 1000))
    return df.rolling(window=win, center=True, min_periods=1).mean()


erp_across_subjects = []
data_joint_modeling_all = []
erp_response_locked_across_subjects = []

beh_files = sorted(PATH_PREPROCESS_DATA_BIDS.glob('sub-*/beh/*_beh.csv'))
eeg_files = sorted(PATH_PREPROCESS_DATA_BIDS.glob('sub-*/eeg/*.vhdr'))
beh_by_sub = {p.parent.parent.name: p for p in beh_files}
eeg_by_sub = {p.parent.parent.name: p for p in eeg_files}

for sub_id in SUBJECT_IDS:
    path_eeg = eeg_by_sub[f'sub-A{sub_id}']
    path_behavior = beh_by_sub[f'sub-A{sub_id}']
    logger.info(f"Processing subject: {sub_id}")

    # === Step 1: load eeg and behavior data ===
    data_behavior = pd.read_csv(path_behavior, sep=',', header=0)
    n_behavior_trials = len(data_behavior)
    logger.info(f"  - Behavioral trials: {n_behavior_trials}")
    assert n_behavior_trials > 0, f"No behavioral data found for subject {sub_id}"

    # === Step 2: Load EEG file ===
    raw = mne.io.read_raw_brainvision(path_eeg, preload=True, verbose='ERROR')
    events, event_id = mne.events_from_annotations(raw, verbose='ERROR')

    # === Step 3: Validate that response event exists ===
    event_key_response = 'Stimulus/S  5'
    assert event_key_response in event_id, f"Response event '{event_key_response}' not found in annotations for {sub_id}"
    event_id_response = {event_key_response: event_id[event_key_response]}
    events_response = events[events[:, 2] == event_id_response[event_key_response]]

    # === Step 4: Ensure behavior data and eeg data have the same trials ===
    n_eeg_trials = len(events_response)
    logger.info(f"  - EEG response-locked trials: {n_eeg_trials}")
    assert n_eeg_trials == n_behavior_trials, f"Trial count mismatch! EEG: {n_eeg_trials}, Behavior: {n_behavior_trials}"

    # === Step 5: Full ERP epoching ===
    epochs_full = mne.Epochs(
        raw=raw,
        events=events_response,
        event_id=event_id_response,
        tmin=ERP_TMIN,
        tmax=ERP_TMAX,
        baseline=(BASELINE_TMIN, BASELINE_TMAX),
        preload=True,
        event_repeated='drop',
        verbose='ERROR',
    )

    epochs_data = epochs_full.get_data()
    scipy.io.savemat(PATH_RESULTS_EPOCH / f'epoch_sub-A{sub_id}.mat', {'EEG_epoch': epochs_data})

    # === Step 6: Extract stimulus-locked and response-locked ERPs ===
    erp_data = epochs_full.get_data(picks=CHANNEL_INDICES)  # shape: (trial, ch, time)
    erp_data_cpp = np.nanmean(erp_data, axis=1)  # shape: (trial, time)
    erp_avg = np.nanmean(erp_data, axis=(0, 1))
    erp_across_subjects.append(erp_avg)  # save for ERP plot which is response-locked

    # === Step 7: Extract AMS, PAMS, SLPS ===
    erp_data_pre4ddm_ams = np.nanmean(erp_data, axis=1)  # shape: (trial, time)
    erp_data_pre4ddm_pams = erp_data_pre4ddm_ams
    erp_data_pre4ddm_slps = smooth(pd.DataFrame(erp_data_pre4ddm_ams), sample_rate=SAMPLING_RATE).to_numpy()
    
    win_duration_ams_slps = abs(T_AMS_SLPS_END - T_AMS_SLPS_START)
    n_samples_ams_slps = int(win_duration_ams_slps * SAMPLING_RATE)

    win_duration_pams = abs(T_PAMS_END - T_PAMS_START)
    n_samples_pams = int(win_duration_pams * SAMPLING_RATE)

    n_time = erp_data_pre4ddm_ams.shape[1]
    data_ams_subj = []
    data_pams_subj = []
    data_slps_subj = []

    # For response-locked data, the time window is relative to response (t=0)
    for trial in range(erp_data_pre4ddm_ams.shape[0]):

        t_start_index = np.abs(ERP_TMIN)
        t_ams_slps_start = int((t_start_index  + T_AMS_SLPS_START) * SAMPLING_RATE)
        t_pams_start = int((t_start_index + T_PAMS_START) * SAMPLING_RATE)
        t_ams_slps_end = t_ams_slps_start + n_samples_ams_slps
        t_pams_end = t_pams_start + n_samples_pams
        
        if t_ams_slps_start < 0 or t_ams_slps_end > n_time:
            data_ams_trial = np.full(n_samples_ams_slps, np.nan)
            data_slps_trial = np.full(n_samples_ams_slps, np.nan)
        else:
            data_ams_trial = erp_data_pre4ddm_ams[trial, t_ams_slps_start:t_ams_slps_end]
            data_slps_trial = erp_data_pre4ddm_slps[trial, t_ams_slps_start:t_ams_slps_end]

        if t_pams_start < 0 or t_pams_end > n_time:
            data_pams_trial = np.full(n_samples_pams, np.nan)
        else:
            data_pams_trial = erp_data_pre4ddm_pams[trial, t_pams_start:t_pams_end]

        data_ams_subj.append(data_ams_trial)
        data_pams_subj.append(data_pams_trial)
        data_slps_subj.append(data_slps_trial)

    data_ams = np.vstack(data_ams_subj)
    data_pams = np.vstack(data_pams_subj)
    data_slps = np.vstack(data_slps_subj)

    # ams: amplitude
    ams = np.nanmean(data_ams, axis=1)
    # pams: peak amplitude
    pams = np.nanmax(data_pams, axis=1)
    # slps: slope
    n_trial = data_slps.shape[0]
    time_vector = np.arange(data_slps.shape[1]) / SAMPLING_RATE
    slps = np.array([np.polyfit(time_vector, data_slps[i, :], deg=1)[0] for i in range(n_trial)])

    if np.all(np.isnan(ams)):
        logger.warning(f"Subject {sub_id}: all AMS values are NaN")
    if np.all(np.isnan(pams)):
        logger.warning(f"Subject {sub_id}: all PAMS values are NaN")
    if np.all(np.isnan(slps)):
        logger.warning(f"Subject {sub_id}: all SLPS values are NaN")

    data_features = pd.DataFrame({'ams': ams, 'pams': pams, 'slps': slps})

    # === Step 8: Calculate features in bin ===
    for feature in ['ams', 'pams', 'slps']:
        quantile_col = f"{feature}_quantile"

        def safe_qcut_original(x):
            x_clean = x.dropna()
            if x_clean.nunique() < 4:
                return pd.Series(['N/A'] * len(x), index=x.index)
            try:
                return pd.qcut(x, q=4, labels=['1st', '2nd', '3rd', '4th'])
            except Exception:
                return pd.Series(['N/A'] * len(x), index=x.index)

        data_features[quantile_col] = safe_qcut_original(data_features[feature])

    for feature in ['ams', 'pams', 'slps']:
        x = data_features[feature]
        if x.dropna().shape[0] < 2:
            data_features[feature] = np.nan
        else:
            data_features[feature] = (x - x.mean()) / x.std()

    for feature in ['ams', 'pams', 'slps']:
        quantile_col = f"{feature}_quantile"
        bin_col = 'slp_bin' if feature == 'slps' else f"{feature[:3]}_bin"
        bin_means = data_features.groupby(quantile_col, dropna=False)[feature].mean().reset_index()
        bin_means.rename(columns={feature: bin_col}, inplace=True)
        data_features = pd.merge(data_features, bin_means, on=quantile_col, how='left')

    # === Step 9: Merge cleaned behavior and EEG features ===
    assert len(data_behavior) == len(data_features), f"Subject {sub_id}: trial counts mismatch after feature extraction"

    data_joint_modeling = pd.concat([data_behavior, data_features], axis=1)
    data_joint_modeling['subj_idx'] = sub_id
    data_joint_modeling_all.append(data_joint_modeling)
    logger.info(f"Subject {sub_id}: features extracted successfully")

if data_joint_modeling_all:
    data_joint_modeling_all = pd.concat(data_joint_modeling_all, axis=0, ignore_index=True)
    data_joint_modeling_all.to_csv(PATH_RESULTS_JOINT / 'data_joint_modeling_all.csv', index=False)
else:
    logger.warning('No subject data was processed; output file was not generated.')